# 银行金融大模型量化实验总 Notebook

本文件统一执行 6.2 AWQ、6.3 GPTQ 和 6.4 SmoothQuant/FP8。

目录：
1. 公共环境和资产
2. AWQ INT4 W4A16
3. GPTQ INT4 W4A16
4. SmoothQuant INT8 / FP8 能力
5. 输出目录和状态检查

完整模型只写入 `banking_llm_project`，分析结果写入 `OptimizationSystem/out`。首次安装后必须重启 Colab，再从公共初始化单元继续。

In [ ]:
%pip -q install -U autoawq gptqmodel==7.5.0 smoothquant optimum accelerate safetensors
%pip -q install -U transformers==5.18.0
print('安装完成。请重启 Colab 会话；重启后跳过本单元。')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc
import importlib.metadata
import json
import sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
sys.path.insert(0, str(PROJECT_DIR))
from src.data.finqa_assets import load_assets
from src.quantization.awq import build_awq_plans, run_awq_plan, collect_awq_evidence, validate_awq_evidence
from src.quantization.gptq import build_gptq_plans, run_gptq_plan, collect_gptq_evidence, validate_gptq_evidence
from src.quantization.int8_fp8 import collect_int8_fp8_evidence, validate_int8_fp8_evidence

assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
assets = load_assets(DATASET_DIR)
print('GPU:', torch.cuda.get_device_name(0))
print('校准集:', len(assets.calibration))
print('Transformers:', importlib.metadata.version('transformers'))

## 6.2 AWQ INT4 W4A16

三个 group size：32、64、128。

In [ ]:
awq_config_path = PROJECT_DIR / 'configs/awq.json'
awq_config = json.loads(awq_config_path.read_text(encoding='utf-8'))
awq_result_dir = PROJECT_DIR / 'out/awq'
awq_model_dir = Path('/content/drive/MyDrive/banking_llm_project/awq')
for plan in build_awq_plans(awq_config):
    manifest = awq_result_dir / plan.plan_id / 'quantized_model_manifest.json'
    if manifest.is_file():
        print('跳过已完成:', plan.plan_id)
        continue
    result = run_awq_plan(
        plan, MODEL_MANIFEST, assets.calibration_prompts,
        awq_result_dir / plan.plan_id, awq_model_dir / plan.plan_id,
        int(awq_config['max_calibration_samples']),
        int(awq_config['max_calibration_seq_len']),
    )
    if result['status'] != 'pass':
        raise RuntimeError(result.get('detail', result))
    gc.collect(); torch.cuda.empty_cache()
awq_evidence = collect_awq_evidence(awq_config_path, CALIBRATION_MANIFEST, MODEL_MANIFEST, awq_result_dir)
awq_evidence_path = awq_result_dir / 'awq_evidence.json'
awq_evidence_path.write_text(json.dumps(awq_evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
awq_report = validate_awq_evidence(awq_evidence_path)
(awq_result_dir / 'validation.json').write_text(json.dumps(awq_report.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(awq_report.to_dict(), ensure_ascii=False, indent=2))

## 6.3 GPTQ INT4 W4A16

与 AWQ 使用相同的校准集、bit 数和 group size。

In [ ]:
gptq_config_path = PROJECT_DIR / 'configs/gptq.json'
gptq_config = json.loads(gptq_config_path.read_text(encoding='utf-8'))
gptq_result_dir = PROJECT_DIR / 'out/gptq'
gptq_model_dir = Path('/content/drive/MyDrive/banking_llm_project/gptq')
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig
for plan in build_gptq_plans(gptq_config):
    manifest = gptq_result_dir / plan.plan_id / 'quantized_model_manifest.json'
    if manifest.is_file():
        print('跳过已完成:', plan.plan_id)
        continue
    result = run_gptq_plan(
        plan, MODEL_MANIFEST, assets.calibration_prompts,
        gptq_result_dir / plan.plan_id, gptq_model_dir / plan.plan_id,
        int(gptq_config['max_calibration_samples']),
        int(gptq_config['max_calibration_seq_len']),
    )
    if result['status'] != 'pass':
        raise RuntimeError(result.get('detail', result))
    gc.collect(); torch.cuda.empty_cache()
gptq_evidence = collect_gptq_evidence(gptq_config_path, CALIBRATION_MANIFEST, MODEL_MANIFEST, gptq_result_dir)
gptq_evidence_path = gptq_result_dir / 'gptq_evidence.json'
gptq_evidence_path.write_text(json.dumps(gptq_evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
gptq_report = validate_gptq_evidence(gptq_evidence_path)
(gptq_result_dir / 'validation.json').write_text(json.dumps(gptq_report.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(gptq_report.to_dict(), ensure_ascii=False, indent=2))

## 6.4 SmoothQuant INT8 / FP8

该节先采集真实后端和硬件能力；没有可用 SmoothQuant API 时保留 `not_run`。

In [ ]:
int8_config_path = PROJECT_DIR / 'configs/int8_fp8.json'
int8_result_dir = PROJECT_DIR / 'out/int8_fp8'
int8_evidence = collect_int8_fp8_evidence(int8_config_path, CALIBRATION_MANIFEST, int8_result_dir)
int8_evidence_path = int8_result_dir / 'int8_fp8_evidence.json'
int8_evidence_path.write_text(json.dumps(int8_evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
int8_report = validate_int8_fp8_evidence(int8_evidence_path)
(int8_result_dir / 'validation.json').write_text(json.dumps(int8_report.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(int8_report.to_dict(), ensure_ascii=False, indent=2))

## 结果检查

AWQ/GPTQ 的质量、回归和 Benchmark 继续使用各自的评测 Notebook；本总 Notebook 负责统一导出和环境证据。

In [ ]:
for name, path in (
    ('AWQ', awq_result_dir / 'validation.json'),
    ('GPTQ', gptq_result_dir / 'validation.json'),
    ('INT8/FP8', int8_result_dir / 'validation.json'),
):
    payload = json.loads(path.read_text(encoding='utf-8'))
    print(name, 'complete=', payload.get('complete'))
print('模型目录：', awq_model_dir, gptq_model_dir)
print('分析目录：', awq_result_dir, gptq_result_dir, int8_result_dir)